# WX-1 자동 학습 (Colab One-Click)
Run all 누르면 전부 자동:
1. 라이브러리 설치
2. (런타임 재시작 안내)
3. 데이터 확장
4. 학습
5. 병합
6. GGUF
7. Drive 저장
8. 다운로드

In [ ]:
GITHUB_USER = "ijeonghu522-stack"
GITHUB_REPO = "wuxicode"
TRAIN_MODEL = "wx-1-coder"
MAX_SAMPLES = 300
print(f"설정: {TRAIN_MODEL}, {MAX_SAMPLES}개")

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q --upgrade "numpy>=2.0" "transformers>=4.46" "trl>=0.12" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44" "datasets>=3.0" "torchao>=0.16"

In [ ]:
print("⚠️ 지금 런타임 → 런타임 다시 시작 누르세요")
print("재시작 후 다음 셀부터 실행")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf wuxicode
!git clone https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git
%cd wuxicode
!ls

In [ ]:
# train_qlora.py 패치
import re

path = "training/train_qlora.py"
s = open(path, encoding="utf-8").read()

# 1. warmup_ratio 라인 제거
s = s.replace('        warmup_ratio=t.get("warmup_ratio", 0.03),\n', '')

# 2. tokenizer=tok -> processing_class=tok
s = s.replace("tokenizer=tok,", "processing_class=tok,")

# 3. TrainingArguments 블록 전체 제거
old_ta = r'    ta = TrainingArguments\([\s\S]*?\)\n'
s = re.sub(old_ta, '', s)

# 4. SFTTrainer 블록 전체 교체
old_sft = r'    trainer = SFTTrainer\([\s\S]*?\)\n'
new_sft = '''    from trl import SFTConfig

    sft_cfg = SFTConfig(
        output_dir=t["output_dir"],
        per_device_train_batch_size=t["per_device_train_batch_size"],
        gradient_accumulation_steps=t["gradient_accumulation_steps"],
        num_train_epochs=t["num_train_epochs"],
        learning_rate=t["learning_rate"],
        warmup_steps=10,
        logging_steps=1,
        save_steps=50,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        max_length=t.get("max_seq_length", 32768),
        dataset_text_field="text",
        packing=False,
    )
    trainer = SFTTrainer(
        model=model,
        processing_class=tok,
        train_dataset=ds,
        args=sft_cfg,
    )
'''
s = re.sub(old_sft, new_sft, s)

open(path, "w", encoding="utf-8").write(s)
print("✓ 패치 완료")

In [ ]:
!python training/auto_build_datasets.py
!wc -l datasets/wx_coder.jsonl

In [ ]:
path = "training/config/wx-1-coder.yaml"
s = open(path, encoding="utf-8").read()
s = s.replace("max_samples: 5", f"max_samples: {MAX_SAMPLES}")
open(path, "w", encoding="utf-8").write(s)
print(f"✓ max_samples: {MAX_SAMPLES}")

In [ ]:
!python training/train_qlora.py --config training/config/wx-1-coder.yaml

In [ ]:
import shutil
shutil.make_archive("wx-1-coder-lora", "zip", "output/wx-1-coder-lora")
shutil.copy("wx-1-coder-lora.zip", "/content/drive/MyDrive/wx-1-coder-lora.zip")
print("✓ LoRA 백업 완료 (Drive)")

In [ ]:
!python training/merge_adapter.py \
    --base Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter output/wx-1-coder-lora \
    --output output/wx-1-coder-merged

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!pip install -q "protobuf>=5.29.0"
!python llama.cpp/convert_hf_to_gguf.py output/wx-1-coder-merged \
    --outfile wx-1-coder-f16.gguf --outtype f16

In [ ]:
!cd llama.cpp && make -j4 llama-quantize
!./llama.cpp/llama-quantize wx-1-coder-f16.gguf wx-1-coder-q4.gguf Q4_K_M

In [ ]:
import shutil
shutil.copy("wx-1-coder-q4.gguf", "/content/drive/MyDrive/wx-1-coder-q4.gguf")
print("✓ Drive 저장 완료")

from google.colab import files
files.download("wx-1-coder-q4.gguf")

In [ ]:
print("=== 완료 ===")
print("Drive: /MyDrive/wx-1-coder-q4.gguf")
print("다운로드: 브라우저 다운로드 폴더")